# Connector: registry

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pecca-core/pecca/blob/main/examples/notebooks/connector_registry.ipynb)

A registry stores model versions, call state and decision logs. The `local` registry writes files; `mlflow` writes runs to an MLflow server. The Pecca API is identical, only `workspace.registry` changes.

In [1]:
%pip install -q pecca requests mlflow

In [2]:
import json, os, re, shutil, subprocess, tempfile, time
from pathlib import Path

import pandas as pd
import pecca
from pecca.core import context
from pecca.data.dataset import Dataset, canonicalize
from pecca.data.synthetic import generate

HERE = Path.cwd()                                   # the notebook's folder (examples/notebooks in the repo)
work = tempfile.mkdtemp()
os.chdir(work)
os.environ["PECCA_HOME"] = os.path.join(work, ".pecca")
os.environ["TQDM_DISABLE"] = "1"
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "1"
import logging
try:
    import mlflow  # MLflow resets its logger level on import, so quiet it afterwards
    logging.getLogger("mlflow").setLevel(logging.ERROR)
except ImportError:
    pass

# Docker-backed cells run only when Docker is reachable and PECCA_SKIP_DOCKER != 1 (CI sets it).
USE_DOCKER = (os.environ.get("PECCA_SKIP_DOCKER") != "1" and shutil.which("docker") is not None
              and subprocess.run(["docker", "info"], capture_output=True).returncode == 0)
print("docker:", "available" if USE_DOCKER else "skipped (PECCA_SKIP_DOCKER=1 or Docker not reachable)")

def sh(*args, check=True):
    return subprocess.run([str(a) for a in args], capture_output=True, text=True, check=check)

import atexit
def cleanup_later(*cmd):
    """Run `cmd` when the kernel exits, so containers are removed even if a cell fails."""
    atexit.register(lambda: subprocess.run([str(c) for c in cmd], capture_output=True))

def free_port():
    import socket
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def wait_for(check, what, seconds=90):
    end = time.time() + seconds
    while time.time() < end:
        try:
            if check():
                return
        except Exception:
            pass
        time.sleep(1)
    raise TimeoutError(f"{what} did not become ready in {seconds}s")

def train_demo(call="route_rfi", rows=3000, candidates=("tfidf_linear",), **kw):
    """Train a small model on the synthetic demo data (LLM answers + human corrections)."""
    df = generate(rows)
    df["text"] = df.email_subject + "\n\n" + df.email_body
    cols = {"input": "text", "llm_output": "llm_rfi", "human_label": "human_rfi", "call_name": {"literal": call.split("/")[-1]}}
    return pecca.train(call, Dataset(canonicalize(df, cols), cols), candidates=list(candidates), **kw)

docker: available


In [3]:
import sys
import requests
from pecca.core.workspace import Workspace

port = free_port()
proc = subprocess.Popen([sys.executable, "-m", "mlflow", "server", "--host", "127.0.0.1", "--port", str(port),
                         "--backend-store-uri", f"sqlite:///{work}/mlflow.db", "--default-artifact-root", f"{work}/artifacts"],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
wait_for(lambda: requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200, "MLflow server")
atexit.register(proc.terminate)
workspaces = {
    "local": Workspace("default", {"workspace": {"registry": "local://./.pecca"}}, Path(work) / ".pecca"),
    "mlflow": Workspace("default", {"workspace": {"registry": f"mlflow://http/127.0.0.1:{port}"}}, Path(work) / ".pecca-mlflow"),
}
for name, ws in workspaces.items():
    r = train_demo(workspace=ws)
    pecca.promote("route_rfi", "shadow", workspace=ws)
    print(f"{name:<7} trained {r.version}: {r.winner}, macro_f1 {r.metric:.2f}")

local   trained v1: tfidf_linear, macro_f1 0.92


mlflow  trained v1: tfidf_linear, macro_f1 0.92


In [4]:
call = {name: context.get_call("route_rfi", ws) for name, ws in workspaces.items()}
pd.DataFrame({name: {"versions": [m.version for m in c.versions()], "mode": c.state().mode, "current_version": c.state().current_version,
                     "registry class": type(c.registry).__name__} for name, c in call.items()})

,local,mlflow
versions,[v1],[v1]
mode,shadow,shadow
current_version,v1,v1
registry class,LocalRegistry,MlflowRegistry


In [5]:
# The same prediction from either registry
x = {"route_rfi": "Card lost\n\nI lost my debit card ending ****1234, please block it."}
{name: pecca.predict("route_rfi", list(x.values()), workspace=ws)[0].label for name, ws in workspaces.items()}

{'local': 'card_lost_stolen', 'mlflow': 'card_lost_stolen'}

In [6]:
proc.terminate(); proc.wait(10)
print("MLflow server stopped")

MLflow server stopped
